# Nexus — EMERALD in JAX on **Craftax-Classic** (A100)

One-click: clone the repo, install the JAX stack, and train the **faithful EMERALD
architecture** (TSSM + MaskGIT world model + imagination actor-critic) on
**Craftax-Classic** — the on-GPU, vmapped JAX reimplementation of Crafter. Because the
env and the whole train loop run on-device (no Python env loop), this is *dramatically*
faster than CPU-bound Crafter. Checkpoints save to your Google Drive and the run resumes
automatically across Colab disconnects.

**Before running:** Runtime → Change runtime type → **A100 GPU** (any GPU works; A100 is fastest).


In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L


In [ ]:
# 2. Get the code
%cd /content
!git clone https://github.com/JuliusSamwer/Nexus_v1.git 2>/dev/null || (cd Nexus_v1 && git pull -q)
%cd /content/Nexus_v1


In [ ]:
# 3. Install the JAX stack. craftax pulls its own jax/gymnax pins; then we make sure
#    jax has CUDA so it runs on the GPU (not CPU).
!pip install -q craftax flax optax
!pip install -q -U "jax[cuda12]"
print('installed')


In [ ]:
# 4. Verify JAX sees the GPU (must list a CudaDevice, not just CPU)
import jax
print('jax', jax.__version__, '| devices:', jax.devices())
assert any('cuda' in d.platform.lower() or d.platform=='gpu' for d in jax.devices()), \
    'No GPU visible to JAX — set Runtime → A100 GPU and re-run cells 3-4.'


In [ ]:
# 5. Mount Drive so checkpoints survive disconnects (run auto-resumes from here)
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/nexus/emerald_jax_craftax'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)


## Train

`config.craftax()` is the **identical** EMERALD architecture/hyperparameters as the
Crafter (torch) baseline — only the environment differs. So anything that helps here is a
candidate to port straight to the Crafter validation run.

Throughput (`env/s`, `grad/s`) prints every log interval — that's the speed number. Bump
`total_env_steps` for a longer run; re-running this cell **resumes** from the last Drive
checkpoint.


In [ ]:
# 6. Train. First call jit-compiles (~1-2 min), then it should fly on the A100.
from emerald_jax import config, train

cfg = config.craftax()
st, ev = train.run(
    cfg,
    total_env_steps = 1_000_000,   # raise for a longer run
    collect_steps   = 64,          # scan-steps per collect (x num_envs env-steps)
    grad_per_collect= 1,
    eval_every_env  = 50_000,
    log_every_env   = 10_000,
    eval_steps      = 1_000,
    ckpt_dir        = CKPT_DIR,
    resume          = True,
)
print('final Crafter score:', ev['score'])


In [ ]:
# 7. Inspect the final per-achievement success rates
import json
print(json.dumps(ev['achievement_rates'], indent=2))


## Notes

- **Resume:** the run checkpoints to Drive every eval; just re-run cell 6 after a
  disconnect and it picks up from `env_step` (the replay buffer re-prefills, which is cheap).
- **Memory:** the on-device replay is uint8 and sized by `cfg.capacity` (rows) × `cfg.num_envs`.
  Default 20000×16 ≈ 3.9GB. Lower `capacity` if you hit OOM on a smaller GPU.
- **Faithfulness:** world-model loss (`observe`) is exact vs `emerald_torch`; online-acting
  context + within-episode window sampling are the documented minor approximations.
